In [ ]:
import os
from pathlib import Path

# Localiza a raiz do projeto de forma robusta (funciona em VS Code, JupyterLab, etc.)
def _find_project_root():
    # VS Code expõe o caminho do notebook nesta variável
    nb_file = globals().get('__vsc_ipynb_file__') or locals().get('__vsc_ipynb_file__')
    if nb_file:
        return Path(nb_file).resolve().parent.parent
    # JupyterLab / linha de comandos
    try:
        import ipynbname
        return ipynbname.path().parent.parent
    except Exception:
        pass
    # Último recurso: working directory atual sobe um nível
    cwd = Path().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
print(f"✓ Working directory: {PROJECT_ROOT}")


# Análise Económica por PTD — Comunidades de Energia Renovável (Aveiro)

Este notebook calcula os indicadores económicos de viabilidade financeira para cada PTD.

**Parâmetros económicos:**
- Custo de instalação: 1200 €/painel
- Preço da energia (autoconsumo): 0,25 €/kWh
- Tarifa excedente injetado na rede (RESP): 0,05 €/kWh
- Horizonte de análise: 25 anos
- Taxa de desconto: 7%
- Degradação anual dos painéis: 0,5%/ano

**Distinção fundamental:**
- **Autoconsumo** = min(produção, consumo) → valorizado a 0,25 €/kWh (energia que deixa de ser comprada à rede)
- **Excedente** = max(produção - consumo, 0) → valorizado a 0,05 €/kWh (injetado na rede RESP)

In [28]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. PARÂMETROS ECONÓMICOS
# ==========================================
CUSTO_POR_PAINEL_EUR     = 1200   # € por painel instalado
PRECO_AUTOCONSUMO        = 0.25    # €/kWh — energia que deixa de ser comprada
PRECO_EXCEDENTE          = 0.05    # €/kWh — tarifa RESP para injeção na rede
HORIZONTE_ANOS           = 25      # anos de análise
TAXA_DESCONTO            = 0.07    # 7% ao ano
DEGRADACAO_ANUAL         = 0.005   # 0,5% de perda de eficiência por ano
POTENCIA_PAINEL_KWP      = 0.450   # kWp por painel
H_ANUAL_PVGIS            = 1860.2  # kWh/m²/ano
PR                       = 0.80    # Performance Ratio

# ==========================================
# 2. CAMINHOS DOS FICHEIROS
# ==========================================
CAMINHO_PTD     = r"data/processed/dimensionamento_realista_paineis_ptd.csv"
CAMINHO_CONSUMO = r"data/processed/comparacao_final_limpa_cp7.csv"
CAMINHO_MAPPING = r"data/processed/potencial_com_mapeamento_ptd.csv"
CAMINHO_OUTPUT  = r"data/processed/analise_economica_ptd.csv"

print("[1/5] A carregar ficheiros de input...")
df_ptd     = pd.read_csv(CAMINHO_PTD, sep=";", encoding="utf-8-sig")
df_consumo = pd.read_csv(CAMINHO_CONSUMO, encoding="utf-8-sig")
df_mapping = pd.read_csv(CAMINHO_MAPPING, sep=";", encoding="utf-8-sig")

df_ptd.columns     = df_ptd.columns.str.strip()
df_consumo.columns = df_consumo.columns.str.strip()
df_mapping.columns = df_mapping.columns.str.strip()

# Recalcular potência e produção reais a partir dos painéis inteiros
df_ptd["potencia_real_kwp"]     = df_ptd["total_paineis_reais"] * POTENCIA_PAINEL_KWP
df_ptd["producao_real_mwh_ano"] = df_ptd["potencia_real_kwp"] * H_ANUAL_PVGIS * PR / 1000

print(f"  PTDs carregados         : {len(df_ptd)}")
print(f"  CP7s com consumo válido : {len(df_consumo)}")
print(f"  Registos de mapeamento  : {len(df_mapping)}")

[1/5] A carregar ficheiros de input...
  PTDs carregados         : 368
  CP7s com consumo válido : 558
  Registos de mapeamento  : 933


In [29]:
LIMIAR_CONSUMO_GWH = 5.0
df_consumo = df_consumo[df_consumo["consumo_anual_kwh"] <= LIMIAR_CONSUMO_GWH * 1e6].copy()
print(f"  CP7s após filtro de grandes consumidores: {len(df_consumo)}")
print(f"  Consumo residencial: {df_consumo['consumo_anual_kwh'].sum()/1e6:.2f} GWh/ano")

  CP7s após filtro de grandes consumidores: 547
  Consumo residencial: 265.55 GWh/ano


In [30]:
print("[2/5] A agregar consumo por PTD...")

df_cons_ptd = pd.merge(
    df_mapping[["cp7", "ptd_id"]],
    df_consumo[["cp7", "consumo_anual_kwh"]],
    on="cp7",
    how="inner"
)

consumo_por_ptd = df_cons_ptd.groupby("ptd_id").agg(
    consumo_anual_kwh=("consumo_anual_kwh", "sum"),
    n_cp7_com_consumo=("cp7", "count")
).reset_index()

print(f"  PTDs com dados de consumo: {len(consumo_por_ptd)}")
print(f"  Consumo total agregado   : {consumo_por_ptd['consumo_anual_kwh'].sum()/1e6:.2f} GWh/ano")

[2/5] A agregar consumo por PTD...


  PTDs com dados de consumo: 290
  Consumo total agregado   : 262.01 GWh/ano


In [31]:
print("[3/5] A construir base de análise por PTD...")

df_base = pd.merge(
    df_ptd[["ptd_id", "total_paineis_reais", "potencia_real_kwp", "producao_real_mwh_ano"]],
    consumo_por_ptd,
    on="ptd_id",
    how="left"
)

# Produção em kWh
df_base["producao_anual_kwh"] = df_base["producao_real_mwh_ano"] * 1000
df_base["consumo_anual_kwh"]  = df_base["consumo_anual_kwh"].fillna(0)

# ==========================================
# DISTINÇÃO AUTOCONSUMO vs EXCEDENTE
# ==========================================
# Autoconsumo: energia produzida que substitui compra à rede
df_base["autoconsumo_kwh"] = np.minimum(df_base["producao_anual_kwh"], df_base["consumo_anual_kwh"])

# Excedente: produção que ultrapassa o consumo local → injetada na rede
df_base["excedente_kwh"] = np.maximum(df_base["producao_anual_kwh"] - df_base["consumo_anual_kwh"], 0)

# Rácio de autossuficiência
df_base["racio_autossuficiencia"] = np.where(
    df_base["consumo_anual_kwh"] > 0,
    df_base["producao_anual_kwh"] / df_base["consumo_anual_kwh"],
    np.nan
)

# Receita/poupança total ano 1 (sem degradação ainda)
df_base["poupanca_ano1_eur"] = (
    df_base["autoconsumo_kwh"] * PRECO_AUTOCONSUMO +
    df_base["excedente_kwh"]   * PRECO_EXCEDENTE
)

print(f"  PTDs com rácio calculável : {df_base['racio_autossuficiencia'].notna().sum()}")
print(f"  PTDs candidatos a CER     : {(df_base['racio_autossuficiencia'] >= 1).sum()} (produção ≥ consumo)")
print(f"  Autoconsumo total ano 1   : {df_base['autoconsumo_kwh'].sum()/1e6:.2f} GWh/ano")
print(f"  Excedente total ano 1     : {df_base['excedente_kwh'].sum()/1e6:.2f} GWh/ano")
print(f"  Poupança total ano 1      : {df_base['poupanca_ano1_eur'].sum()/1e6:.2f} M€/ano")

[3/5] A construir base de análise por PTD...
  PTDs com rácio calculável : 290
  PTDs candidatos a CER     : 149 (produção ≥ consumo)
  Autoconsumo total ano 1   : 170.19 GWh/ano
  Excedente total ano 1     : 106.34 GWh/ano
  Poupança total ano 1      : 47.86 M€/ano


In [32]:
print("[4/5] A calcular indicadores económicos por PTD...")

df_base["custo_instalacao_eur"] = df_base["total_paineis_reais"] * CUSTO_POR_PAINEL_EUR

def calcular_indicadores(row):
    investimento   = row["custo_instalacao_eur"]
    autoconsumo_b  = row["autoconsumo_kwh"]  # base ano 1
    excedente_b    = row["excedente_kwh"]     # base ano 1

    if investimento == 0 or (autoconsumo_b + excedente_b) == 0:
        return pd.Series({"val_eur": np.nan, "tir_pct": np.nan, "payback_anos": np.nan})

    # Fluxos de caixa com degradação anual
    fluxos = [-investimento]
    for t in range(1, HORIZONTE_ANOS + 1):
        fator = (1 - DEGRADACAO_ANUAL) ** t
        receita_t = (
            autoconsumo_b * fator * PRECO_AUTOCONSUMO +
            excedente_b   * fator * PRECO_EXCEDENTE
        )
        fluxos.append(receita_t)

    # VAL
    val = sum(f / (1 + TAXA_DESCONTO) ** t for t, f in enumerate(fluxos))

    # TIR (bissecção numérica)
    def npv(rate):
        return sum(f / (1 + rate) ** t for t, f in enumerate(fluxos))

    try:
        lo, hi = -0.999, 10.0
        tir = np.nan
        if npv(lo) * npv(hi) < 0:
            for _ in range(200):
                mid = (lo + hi) / 2
                if npv(mid) > 0:
                    lo = mid
                else:
                    hi = mid
            tir = (lo + hi) / 2
    except Exception:
        tir = np.nan

    # Payback simples (sem desconto)
    acumulado = -investimento
    payback = np.nan
    for t in range(1, HORIZONTE_ANOS + 1):
        fator = (1 - DEGRADACAO_ANUAL) ** t
        acumulado += (
            autoconsumo_b * fator * PRECO_AUTOCONSUMO +
            excedente_b   * fator * PRECO_EXCEDENTE
        )
        if acumulado >= 0:
            payback = t
            break

    return pd.Series({"val_eur": val, "tir_pct": tir * 100 if not np.isnan(tir) else np.nan, "payback_anos": payback})

indicadores = df_base.apply(calcular_indicadores, axis=1)
df_final = pd.concat([df_base, indicadores], axis=1)

print("  Cálculo concluído.")

[4/5] A calcular indicadores económicos por PTD...
  Cálculo concluído.


In [33]:
print("[5/5] A exportar resultados e apresentar resumo...")

df_final.to_csv(CAMINHO_OUTPUT, index=False, sep=";", encoding="utf-8-sig")
print(f"  ✓ Ficheiro guardado em '{CAMINHO_OUTPUT}'")

df_viavel    = df_final[df_final["val_eur"].notna()]
df_positivo  = df_viavel[df_viavel["val_eur"] > 0]
df_candidatos = df_final[df_final["racio_autossuficiencia"] >= 1]

print("")
print("=======================================================")
print("           RESUMO ECONÓMICO GLOBAL")
print("=======================================================")
print(f" PTDs analisados                     : {len(df_final)}")
print(f" PTDs com análise económica válida   : {len(df_viavel)}")
print(f" PTDs candidatos a CER (prod≥cons)   : {len(df_candidatos)}")
print(f" PTDs com VAL positivo (viáveis)     : {len(df_positivo)}")
print("")
print(f" Investimento total estimado         : {df_final['custo_instalacao_eur'].sum()/1e6:.2f} M€")
print(f" Autoconsumo total (ano 1)           : {df_final['autoconsumo_kwh'].sum()/1e6:.2f} GWh/ano")
print(f" Excedente total (ano 1)             : {df_final['excedente_kwh'].sum()/1e6:.2f} GWh/ano")
print(f" Poupança total ano 1                : {df_final['poupanca_ano1_eur'].sum()/1e6:.2f} M€/ano")
print("")
print(f" VAL mediano (PTDs viáveis)          : {df_viavel['val_eur'].median()/1e3:.1f} k€")
print(f" TIR mediana (PTDs viáveis)          : {df_viavel['tir_pct'].median():.1f}%")
print(f" Payback mediano (PTDs viáveis)      : {df_viavel['payback_anos'].median():.1f} anos")
print("=======================================================")

print("")
print("Top 10 PTDs com maior VAL:")
colunas = ["ptd_id", "total_paineis_reais", "potencia_real_kwp",
           "autoconsumo_kwh", "excedente_kwh", "racio_autossuficiencia",
           "custo_instalacao_eur", "poupanca_ano1_eur", "val_eur", "tir_pct", "payback_anos"]
print(df_final.nlargest(10, "val_eur")[colunas].to_string(index=False))

[5/5] A exportar resultados e apresentar resumo...
  ✓ Ficheiro guardado em 'C:\Documentos\UA_Mestrado\1UA\2 semestre\seminario\Milestone IV\numero de paines a se instalar por PTD\02_Dados_Processados\analise_economica_ptd.csv'

           RESUMO ECONÓMICO GLOBAL
 PTDs analisados                     : 368
 PTDs com análise económica válida   : 368
 PTDs candidatos a CER (prod≥cons)   : 149
 PTDs com VAL positivo (viáveis)     : 226

 Investimento total estimado         : 495.51 M€
 Autoconsumo total (ano 1)           : 170.19 GWh/ano
 Excedente total (ano 1)             : 106.34 GWh/ano
 Poupança total ano 1                : 47.86 M€/ano

 VAL mediano (PTDs viáveis)          : 148.0 k€
 TIR mediana (PTDs viáveis)          : 9.3%
 Payback mediano (PTDs viáveis)      : 8.0 anos

Top 10 PTDs com maior VAL:
 ptd_id  total_paineis_reais  potencia_real_kwp  autoconsumo_kwh  excedente_kwh  racio_autossuficiencia  custo_instalacao_eur  poupanca_ano1_eur      val_eur   tir_pct  payback_anos
   

## Notas Metodológicas

**Autoconsumo vs. Excedente:** A receita anual de cada PTD é composta por duas componentes — a energia autoconsumida (valorizada ao preço de mercado evitado de 0,25 €/kWh) e o excedente injetado na rede (remunerado à tarifa RESP de 0,05 €/kWh). Esta distinção é fundamental para evitar a sobrevalorização do retorno em PTDs com produção muito superior ao consumo local.

**Degradação:** A perda de eficiência de 0,5%/ano é aplicada de forma composta sobre ambas as componentes (autoconsumo e excedente), refletindo o envelhecimento natural dos módulos mono-Si.

**Custos de O&M:** Este modelo não inclui custos de operação e manutenção anuais. Para uma análise mais conservadora, recomenda-se adicionar ~1% do investimento inicial por ano.

**PTDs sem consumo emparelhado:** PTDs sem dados de consumo da E-Redes têm autoconsumo=0 e toda a produção é tratada como excedente, o que subestima o seu retorno real.

In [34]:
df_consumo = pd.read_csv(r"data/processed/comparacao_final_limpa_cp7.csv", encoding="utf-8-sig")
print(df_consumo["consumo_anual_kwh"].describe())
print(f"\nTotal: {df_consumo['consumo_anual_kwh'].sum()/1e6:.2f} GWh/ano")
print(f"Mediana: {df_consumo['consumo_anual_kwh'].median()/1e3:.1f} MWh/ano por CP7")

count    5.580000e+02
mean     7.654156e+05
std      2.619237e+06
min      5.896568e+04
25%      2.073790e+05
50%      3.402934e+05
75%      5.902425e+05
max      4.522472e+07
Name: consumo_anual_kwh, dtype: float64

Total: 427.10 GWh/ano
Mediana: 340.3 MWh/ano por CP7


In [35]:
df_consumo = pd.read_csv(r"data/processed/comparacao_final_limpa_cp7.csv", encoding="utf-8-sig")
print("CP7s com consumo > 5 GWh/ano:")
print(df_consumo[df_consumo["consumo_anual_kwh"] > 5e6][["cp7", "consumo_anual_kwh", "n_registos"]].sort_values("consumo_anual_kwh", ascending=False).to_string(index=False))

CP7s com consumo > 5 GWh/ano:
     cp7  consumo_anual_kwh  n_registos
3800-536       4.522472e+07       17544
3800-055       2.983753e+07       17544
3800-587       1.655393e+07       17544
3810-498       1.377765e+07       17544
3810-140       1.017065e+07       17544
3800-525       9.905595e+06       17544
3810-168       9.430810e+06       17544
3810-783       9.138240e+06       17544
3810-106       6.306733e+06       17544
3800-043       5.633717e+06       17544
3810-536       5.573636e+06       17544


In [36]:
LIMIAR_CONSUMO_GWH = 5.0  # excluir CP7 com consumo > 5 GWh/ano
df_consumo_filtrado = df_consumo[df_consumo["consumo_anual_kwh"] <= LIMIAR_CONSUMO_GWH * 1e6]
print(f"CP7s removidos: {len(df_consumo) - len(df_consumo_filtrado)}")
print(f"Consumo residual: {df_consumo_filtrado['consumo_anual_kwh'].sum()/1e6:.2f} GWh/ano")

CP7s removidos: 11
Consumo residual: 265.55 GWh/ano
